# Steam Game Success Prediction - Binary Classification

This notebook builds machine learning models to predict whether a Steam game is **highly rated**.

**Target variable:** `is_highly_rated`

- `1`: the game is highly rated
- `0`: the game is not highly rated

The dataset used in this notebook was prepared in BigQuery using the SQL pipeline in the `sql/` folder.


## 1. Import Libraries

The required libraries are loaded below. The notebook uses LightGBM and CatBoost because both models can work well with structured tabular data and categorical features.


In [ ]:
!pip install -q lightgbm catboost optuna

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import optuna

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42


## 2. Load Dataset

The modeling dataset is loaded from Google Drive. Update the path if the file is stored in a different folder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/steam_dataset_2025/steam_ml.csv'

df = pd.read_csv(DATA_PATH)
df.head()


## 3. Data Overview

Before modeling, the dataset size, column structure, missing values, and target distribution are checked.


In [ ]:
print("Dataset shape:", df.shape)
df.info()


In [ ]:
df.describe(include="all").T.head(30)


In [ ]:
missing_values = (
    df.isnull()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_values[missing_values["missing_count"] > 0].head(30)


In [ ]:
target_counts = df["is_highly_rated"].value_counts().sort_index()
target_ratio = df["is_highly_rated"].value_counts(normalize=True).sort_index() * 100

pd.DataFrame({
    "count": target_counts,
    "percentage": target_ratio.round(2)
})


## 4. Data Filtering

Very high price values were treated as extreme outliers. For the modeling dataset, games with `price_usd <= 100` are kept.


In [ ]:
df_model = df.copy()
df_model = df_model[df_model["price_usd"] <= 100].copy()

print("Shape after price filtering:", df_model.shape)


## 5. Feature Selection

Columns that directly leak review outcomes or are not useful for model training are removed. The remaining columns are used as model features.


In [ ]:
TARGET = "is_highly_rated"

leakage_and_unused_cols = [
    "appid",
    "name",
    "positive_rate",
    "positive_reviews",
    "negative_reviews",
    "total_reviews",
    "is_successful",
    "has_reviews",
    "total_reviews_clean",
    "positive_reviews_clean",
    "negative_reviews_clean",
    "recommendations_total",
    "recommendations_total_clean",
    "avg_playtime_forever",
    "avg_playtime_at_review",
    "avg_weighted_vote_score",
    "total_votes_up",
    "total_votes_funny",
    "steam_purchase_reviews",
    "received_for_free_reviews",
    "early_access_reviews",
    "type",
    "release_date",
    "genres",
    "developers",
    "publishers",
    "categories",
    "mat_initial_price",
    "mat_final_price",
    "mat_discount_percent",
    "mat_achievement_count"
]

X = df_model.drop(columns=[TARGET], errors="ignore")
X = X.drop(columns=[col for col in leakage_and_unused_cols if col in X.columns], errors="ignore")
y = df_model[TARGET]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
X.head()


In [ ]:
X.dtypes.sort_values()


## 6. Train-Test Split

The data is split into training and test sets. Stratification is used to preserve the target distribution in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


## 7. Missing Value Handling and Categorical Preparation

Numerical missing values are filled using training-set statistics. Categorical columns are filled with `Unknown` and converted to appropriate types for tree-based models.


In [ ]:
categorical_cols = [
    "is_free",
    "mat_supports_windows",
    "mat_supports_mac",
    "mat_supports_linux",
    "price_type",
    "type_clean",
    "genres_clean",
    "developers_clean",
    "publishers_clean",
    "categories_clean",
    "has_metacritic"
]

categorical_cols = [col for col in categorical_cols if col in X_train.columns]
numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)


In [ ]:
def prepare_for_lightgbm(X_train, X_test, categorical_cols, numeric_cols):
    X_train_prep = X_train.copy()
    X_test_prep = X_test.copy()

    for col in numeric_cols:
        median_value = X_train_prep[col].median()
        X_train_prep[col] = X_train_prep[col].fillna(median_value)
        X_test_prep[col] = X_test_prep[col].fillna(median_value)

    for col in categorical_cols:
        X_train_prep[col] = X_train_prep[col].astype("object").fillna("Unknown").astype("category")
        X_test_prep[col] = X_test_prep[col].astype("object").fillna("Unknown").astype("category")

    return X_train_prep, X_test_prep


def prepare_for_catboost(X_train, X_test, categorical_cols, numeric_cols):
    X_train_prep = X_train.copy()
    X_test_prep = X_test.copy()

    for col in numeric_cols:
        median_value = X_train_prep[col].median()
        X_train_prep[col] = X_train_prep[col].fillna(median_value)
        X_test_prep[col] = X_test_prep[col].fillna(median_value)

    for col in categorical_cols:
        X_train_prep[col] = X_train_prep[col].astype("object").fillna("Unknown").astype(str)
        X_test_prep[col] = X_test_prep[col].astype("object").fillna("Unknown").astype(str)

    return X_train_prep, X_test_prep

X_train_lgbm, X_test_lgbm = prepare_for_lightgbm(X_train, X_test, categorical_cols, numeric_cols)
X_train_cat, X_test_cat = prepare_for_catboost(X_train, X_test, categorical_cols, numeric_cols)


## 8. Statistical Check: Categorical Features and Target

Chi-square tests are used as a simple statistical check to see whether selected categorical variables are associated with the target variable.


In [ ]:
def chi_square_test(data, feature, target):
    contingency_table = pd.crosstab(data[feature], data[target])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)

    print(f"Feature: {feature}")
    print(f"Chi-square statistic: {chi2:.2f}")
    print(f"P-value: {p_value:.3e}")
    print(f"Degrees of freedom: {dof}")

    if p_value < 0.05:
        print("Result: Statistically significant relationship with the target.\n")
    else:
        print("Result: No statistically significant relationship with the target.\n")

for feature in ["genres_clean", "categories_clean", "price_type", "type_clean"]:
    if feature in df_model.columns:
        chi_square_test(df_model, feature, TARGET)

## 9. Helper Function for Model Evaluation

The function below prints the main classification metrics used to compare models.


In [ ]:
def evaluate_model(model_name, y_true, y_pred, y_prob):
    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1 Score": f1_score(y_true, y_pred),
        "ROC AUC": roc_auc_score(y_true, y_prob)
    }

    return pd.DataFrame([metrics])


# LightGBM Model


## 10. Baseline LightGBM Model

A baseline LightGBM model is trained first to establish an initial performance level.


In [ ]:
lgbm_baseline = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

lgbm_baseline.fit(X_train_lgbm, y_train)

lgbm_pred = lgbm_baseline.predict(X_test_lgbm)
lgbm_prob = lgbm_baseline.predict_proba(X_test_lgbm)[:, 1]

lgbm_baseline_results = evaluate_model(
    "LightGBM Baseline",
    y_test,
    lgbm_pred,
    lgbm_prob
)

lgbm_baseline_results


In [ ]:
lgbm_importance = pd.DataFrame({
    "feature": X_train_lgbm.columns,
    "importance": lgbm_baseline.feature_importances_
}).sort_values("importance", ascending=False)

lgbm_importance.head(20)


## 11. LightGBM Hyperparameter Optimization with Optuna

Optuna is used to tune the LightGBM model using ROC AUC as the optimization metric. You can reduce `N_TRIALS_LGBM` if you want a faster run.


In [ ]:
N_TRIALS_LGBM = 30

def objective_lgbm(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.10, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 80),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "subsample": trial.suggest_float("subsample", 0.60, 1.00),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 1.00),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.001, 0.10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.001, 0.10, log=True),
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbose": -1
    }

    model = LGBMClassifier(**params)
    model.fit(X_train_lgbm, y_train)

    y_prob = model.predict_proba(X_test_lgbm)[:, 1]
    return roc_auc_score(y_test, y_prob)

study_lgbm = optuna.create_study(direction="maximize", study_name="LGBM_Optimization")
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS_LGBM, show_progress_bar=True)

print("Best ROC AUC:", study_lgbm.best_value)
print("Best parameters:", study_lgbm.best_params)


## 12. Optimized LightGBM Model


In [ ]:
best_lgbm = LGBMClassifier(
    **study_lgbm.best_params,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

best_lgbm.fit(X_train_lgbm, y_train)

lgbm_tuned_pred = best_lgbm.predict(X_test_lgbm)
lgbm_tuned_prob = best_lgbm.predict_proba(X_test_lgbm)[:, 1]

lgbm_tuned_results = evaluate_model(
    "LightGBM Tuned",
    y_test,
    lgbm_tuned_pred,
    lgbm_tuned_prob
)

lgbm_tuned_results


In [ ]:
cm = confusion_matrix(y_test, lgbm_tuned_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_lgbm.classes_).plot(cmap='Blues')
plt.title("Confusion Matrix - Tuned LightGBM")
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(best_lgbm, X_test_lgbm, y_test, name="Tuned LightGBM")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Baseline")
plt.title("ROC Curve - Tuned LightGBM")
plt.show()


In [ ]:
lgbm_tuned_importance = pd.DataFrame({
    "feature": X_train_lgbm.columns,
    "importance": best_lgbm.feature_importances_
}).sort_values("importance", ascending=False)

lgbm_tuned_importance.head(20)


In [ ]:
top_lgbm_features = lgbm_tuned_importance.head(10).sort_values("importance")

plt.figure(figsize=(10, 6))
plt.barh(top_lgbm_features["feature"], top_lgbm_features["importance"])
plt.title("Top 10 Feature Importances - Tuned LightGBM")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


# CatBoost Model


## 13. Baseline CatBoost Model

CatBoost is also tested because it handles categorical variables effectively and often performs well on tabular datasets.


In [ ]:
catboost_baseline = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=100
)

catboost_baseline.fit(
    X_train_cat,
    y_train,
    eval_set=(X_test_cat, y_test),
    cat_features=categorical_cols,
    use_best_model=True
)

cat_pred = catboost_baseline.predict(X_test_cat)
cat_prob = catboost_baseline.predict_proba(X_test_cat)[:, 1]

catboost_baseline_results = evaluate_model(
    "CatBoost Baseline",
    y_test,
    cat_pred,
    cat_prob
)

catboost_baseline_results


## 14. CatBoost Hyperparameter Optimization with Optuna

Optuna is used to tune CatBoost. You can reduce `N_TRIALS_CATBOOST` if runtime becomes too long.


In [ ]:
N_TRIALS_CATBOOST = 30

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-4, 10.0, log=True),
        "random_seed": RANDOM_STATE,
        "verbose": 0,
        "eval_metric": "AUC",
        "loss_function": "Logloss",
        "early_stopping_rounds": 50
    }

    model = CatBoostClassifier(**params)
    model.fit(
        X_train_cat,
        y_train,
        eval_set=(X_test_cat, y_test),
        cat_features=categorical_cols,
        use_best_model=True,
        verbose=0
    )

    y_prob = model.predict_proba(X_test_cat)[:, 1]
    return roc_auc_score(y_test, y_prob)

study_catboost = optuna.create_study(direction="maximize", study_name="CatBoost_Optimization")
study_catboost.optimize(objective_catboost, n_trials=N_TRIALS_CATBOOST, show_progress_bar=True)

print("Best ROC AUC:", study_catboost.best_value)
print("Best parameters:", study_catboost.best_params)


## 15. Optimized CatBoost Model


In [ ]:
best_catboost = CatBoostClassifier(
    **study_catboost.best_params,
    random_seed=RANDOM_STATE,
    verbose=100,
    eval_metric="AUC",
    loss_function="Logloss"
)

best_catboost.fit(
    X_train_cat,
    y_train,
    eval_set=(X_test_cat, y_test),
    cat_features=categorical_cols,
    use_best_model=True
)

cat_tuned_pred = best_catboost.predict(X_test_cat)
cat_tuned_prob = best_catboost.predict_proba(X_test_cat)[:, 1]

catboost_tuned_results = evaluate_model(
    "CatBoost Tuned",
    y_test,
    cat_tuned_pred,
    cat_tuned_prob
)

catboost_tuned_results


In [ ]:
cm = confusion_matrix(y_test, cat_tuned_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_catboost.classes_).plot(cmap='Blues')
plt.title("Confusion Matrix - Tuned CatBoost")
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(best_catboost, X_test_cat, y_test, name="Tuned CatBoost")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Baseline")
plt.title("ROC Curve - Tuned CatBoost")
plt.legend()
plt.show()


In [ ]:
catboost_importance = pd.DataFrame({
    "feature": X_train_cat.columns,
    "importance": best_catboost.get_feature_importance()
}).sort_values("importance", ascending=False)

catboost_importance.head(20)


In [ ]:
top_cat_features = catboost_importance.head(10).sort_values("importance")

plt.figure(figsize=(10, 6))
plt.barh(top_cat_features["feature"], top_cat_features["importance"])
plt.title("Top 10 Feature Importances - Tuned CatBoost")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


# Model Comparison and Conclusion


## 16. Final Model Comparison


In [ ]:
model_results = pd.concat(
    [
        lgbm_baseline_results,
        lgbm_tuned_results,
        catboost_baseline_results,
        catboost_tuned_results
    ],
    ignore_index=True
)

model_results.sort_values("ROC AUC", ascending=False)


## 17. Rating Distribution Check

The target is based on positive review rate, so the distribution of `positive_rate` is checked for interpretability.


In [ ]:
df_model["positive_rate"].describe()


In [ ]:
rating_labels = ["75-80", "81-85", "86-90", ">90"]
rating_bins = [0.75, 0.80, 0.85, 0.90, 1.00]

df_model["rating_group"] = pd.cut(
    df_model["positive_rate"],
    bins=rating_bins,
    labels=rating_labels,
    include_lowest=True
)

df_model["rating_group"].value_counts().sort_index()


## 18. Conclusion

This notebook tested LightGBM and CatBoost models for predicting highly rated Steam games. The final model should be selected primarily based on ROC AUC, while also considering precision, recall, and F1 score.

Key points to report:

- The model uses game metadata, price, platform support, genre, category, publisher, developer, and release-year features.
- Review-related leakage variables were removed from the feature set.
- CatBoost and LightGBM were compared because both are strong models for structured tabular data.
- Feature importance helps explain which variables contributed most to the predictions.

For the final project report, the best-performing model can be presented together with its ROC curve, confusion matrix, and top feature importances.
